# Llama 3.1 8B — IB Quantization Pipeline (Windows + CUDA)

**Machine:** Windows · RTX 4000 Ada · 21.5GB VRAM · 137GB RAM

| Cell | What it does | Time |
|---|---|---|
| 1 | Install packages + login + config | 2 min |
| 2 | Load GSM8K dataset | 1 min |
| 3 | Load Llama 3.1 8B | 3 min |
| 4 | Tokenize | 2 min |
| 5 | Utility functions | instant |
| 6 | EXP 1 — KL-D map | 20 min |
| 7 | EXP 1 — T-selection scores | 8 min |
| 8 | EXP 4 — Information plane | 8 min |
| 9 | Compute β and bit allocation | instant |
| 10 | Run all experiments | 6 min |
| 11 | Results summary + save | instant |

**Total: ~50 minutes**

---
## Cell 1 — Install, Login, Config

In [3]:
# ── Install everything needed ─────────────────────────────────────
import subprocess
subprocess.run([
    'pip', 'install',
    'transformers', 'datasets', 'huggingface_hub',
    'accelerate', 'scipy', 'psutil', '-q'
], check=True)
print('Packages installed ✓')

# ── Login ─────────────────────────────────────────────────────────
from huggingface_hub import login, get_token
login(token="YOUR_HUGGINGFACE_TOKEN")
TOKEN = get_token()
print(f'Logged in ✓  Token: {TOKEN[:12]}...')

# ── Imports ───────────────────────────────────────────────────────
import torch
import numpy as np
import psutil
import warnings
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────
MODEL_NAME    = 'meta-llama/Llama-3.1-8B-Instruct'
DEVICE        = 'cuda'           # RTX 4000 Ada
MODEL_DTYPE   = torch.float16    # float16 for NVIDIA — faster than bfloat16
N_CALIBRATION = 500
N_VALIDATION  = 100
MAX_LEN       = 128
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── System check ──────────────────────────────────────────────────
vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
ram   = psutil.virtual_memory().total / 1e9
avail = psutil.virtual_memory().available / 1e9

print(f'\nGPU:       {torch.cuda.get_device_name(0)}')
print(f'VRAM:      {vram:.1f} GB')
print(f'RAM total: {ram:.1f} GB')
print(f'RAM free:  {avail:.1f} GB')
print(f'Model:     {MODEL_NAME}')
print(f'Dtype:     {MODEL_DTYPE}')
print()
if vram >= 16:
    print('✓ VRAM sufficient for Llama 3.1 8B in float16 (~16GB needed)')
else:
    print('⚠ VRAM may be tight — model will use GPU+CPU offloading')

Packages installed ✓
Logged in ✓  Token: hf_RGuESVULB...

GPU:       NVIDIA RTX 4000 Ada Generation
VRAM:      21.5 GB
RAM total: 137.1 GB
RAM free:  112.0 GB
Model:     meta-llama/Llama-3.1-8B-Instruct
Dtype:     torch.float16

✓ VRAM sufficient for Llama 3.1 8B in float16 (~16GB needed)


---
## Cell 2 — Load GSM8K Dataset

In [5]:
import requests
import json

print('Loading GSM8K...')

url_train = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/train.jsonl"
url_test  = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"

train_raw = [json.loads(line) for line in requests.get(url_train).text.strip().split('\n')]
test_raw  = [json.loads(line) for line in requests.get(url_test).text.strip().split('\n')]

print(f'Train size: {len(train_raw)}, Test size: {len(test_raw)}')

def extract_answer(answer_text):
    """
    GSM8K answers end with '#### 42' where 42 is the final number.
    Extract the final number as the completion.
    """
    parts = answer_text.split('####')
    if len(parts) > 1:
        return ' ' + parts[-1].strip()
    return ' ' + answer_text.strip()[:30]

def build_pairs(data, n):
    """
    Build (prompt, completion) pairs from GSM8K data.
    Prompt = question text.
    Completion = the final numerical answer.
    """
    pairs = []
    for item in data[:n]:
        prompt     = item['question'].strip()
        completion = extract_answer(item['answer'])
        pairs.append((prompt, completion))
    return pairs

# Use test split for calibration (500) and train split for validation (100)
# They are different so there is no data leakage
CALIBRATION = build_pairs(test_raw,  N_CALIBRATION)
VALIDATION  = build_pairs(train_raw, N_VALIDATION)

print(f'Calibration: {len(CALIBRATION)} pairs')
print(f'Validation:  {len(VALIDATION)} pairs')
print(f'\nExample prompt:     {CALIBRATION[0][0][:80]}...')
print(f'Example completion: {CALIBRATION[0][1]}')

Loading GSM8K...
Train size: 7473, Test size: 1319
Calibration: 500 pairs
Validation:  100 pairs

Example prompt:     Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
Example completion:  18


---
## Cell 3 — Load Llama 3.1 8B Model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {MODEL_NAME}...')
print('This takes 2-3 minutes. VRAM will rise to ~16GB.')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=TOKEN
)
tokenizer.pad_token = tokenizer.eos_token

model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype       = MODEL_DTYPE,
    device_map        = 'auto',     # auto handles multi-GPU or GPU+CPU if needed
    token             = TOKEN,
    low_cpu_mem_usage = True,
)
model_fp.eval()

N_LAYERS   = model_fp.config.num_hidden_layers
HIDDEN_DIM = model_fp.config.hidden_size
VOCAB_SIZE = model_fp.config.vocab_size
N_PARAMS   = sum(p.numel() for p in model_fp.parameters())

print(f'\nLayers:     {N_LAYERS}')
print(f'Hidden dim: {HIDDEN_DIM}')
print(f'Vocab:      {VOCAB_SIZE:,}')
print(f'Parameters: {N_PARAMS:,}')
print(f'VRAM used:  {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('\nModel loaded ✓')

Loading meta-llama/Llama-3.1-8B-Instruct...
This takes 2-3 minutes. VRAM will rise to ~16GB.


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Layers:     32
Hidden dim: 4096
Vocab:      128,256
Parameters: 8,030,261,248
VRAM used:  16.1 GB

Model loaded ✓


---
## Cell 4 — Tokenize Dataset

In [ ]:
def tokenize_pairs(data, max_len=MAX_LEN):
    """
    Tokenize (prompt, completion) pairs.
    full_enc   = full text — used for KL-D and activations
    prompt_enc = prompt only — used to find completion start for accuracy
    """
    full_enc = tokenizer(
        [p + c for p, c in data],
        return_tensors='pt', padding=True,
        truncation=True, max_length=max_len
    ).to(DEVICE)

    prompt_enc = tokenizer(
        [p for p, c in data],
        return_tensors='pt', padding=True,
        truncation=True, max_length=max_len
    ).to(DEVICE)

    return full_enc, prompt_enc

print('Tokenizing...')
cal_enc, cal_prompt_enc = tokenize_pairs(CALIBRATION)
val_enc, val_prompt_enc = tokenize_pairs(VALIDATION)

print(f'Calibration shape: {cal_enc["input_ids"].shape}')
print(f'Validation shape:  {val_enc["input_ids"].shape}')
print('Tokenized ✓')

Tokenizing...
Calibration shape: torch.Size([500, 128])
Validation shape:  torch.Size([100, 117])
Tokenized ✓


---
## Cell 5 — All Utility Functions

**Key difference from GPT-2 pipeline:**
- `get_layer_linears` uses Llama structure: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`
- `store_fp32_outputs` runs FP32 model ONCE and caches logits — avoids copying 8B model 32 times
- All quantization is in-place — no deepcopy needed

In [8]:
# ── Get linear layers for a transformer block ─────────────────────
def get_layer_linears(model, layer_idx):
    """
    Llama 3.1 block has 7 linear layers:
      q_proj, k_proj, v_proj  — attention projections
      o_proj                  — attention output
      gate_proj, up_proj      — MLP SwiGLU gate
      down_proj               — MLP output
    """
    block = model.model.layers[layer_idx]
    return [
        ('q_proj',    block.self_attn.q_proj),
        ('k_proj',    block.self_attn.k_proj),
        ('v_proj',    block.self_attn.v_proj),
        ('o_proj',    block.self_attn.o_proj),
        ('gate_proj', block.mlp.gate_proj),
        ('up_proj',   block.mlp.up_proj),
        ('down_proj', block.mlp.down_proj),
    ]


# ── INT8 quantization in-place ────────────────────────────────────
def quantize_int8_inplace(layer):
    """Symmetric per-tensor INT8. Maps weights to 256 levels."""
    with torch.no_grad():
        W = layer.weight.data.float()
        s = W.abs().max() / 127.0 + 1e-8
        layer.weight.data = ((W/s).round().clamp(-127,127)*s).to(layer.weight.dtype)


# ── INT4 quantization in-place ────────────────────────────────────
def quantize_int4_inplace(layer):
    """Symmetric per-tensor INT4. Maps weights to 16 levels."""
    with torch.no_grad():
        W = layer.weight.data.float()
        s = W.abs().max() / 7.0 + 1e-8
        layer.weight.data = ((W/s).round().clamp(-7,7)*s).to(layer.weight.dtype)


# ── Store FP32 outputs ONCE — reuse for all 32 layer experiments ──
def store_fp32_outputs(model, enc):
    """
    Run FP32 model once on all prompts. Store logits on CPU.
    This is the key efficiency trick:
      Instead of copying 8B model 32 times (would need 256GB RAM),
      we run FP32 once and reuse the stored outputs.
    """
    print('  Storing FP32 outputs (runs once, reused for all layers)...')
    fp32_logits = []
    n = enc['input_ids'].shape[0]

    with torch.no_grad():
        for i in range(n):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)
            logits = model(input_ids=ids, attention_mask=mask).logits
            fp32_logits.append(logits.cpu())  # store on CPU to free VRAM

            if (i+1) % 100 == 0:
                print(f'    {i+1}/{n} done')
                torch.cuda.empty_cache()  # free any fragmented VRAM

    print(f'  Stored {len(fp32_logits)} logit tensors on CPU')
    return fp32_logits


# ── KL-Divergence using stored FP32 outputs ───────────────────────
def kl_from_stored(model_q, enc, fp32_logits):
    """
    Compare quantized model output against stored FP32 logits.
    KL(P||Q) averaged across all prompts and token positions.
    """
    kl_vals = []
    n = enc['input_ids'].shape[0]

    with torch.no_grad():
        for i in range(n):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)

            lq = model_q(input_ids=ids, attention_mask=mask).logits
            lp = fp32_logits[i].to(DEVICE)

            # Use float32 for KL computation — float16 can cause numerical issues
            p  = torch.softmax(lp.float(), dim=-1).clamp(1e-9, 1.0)
            q  = torch.softmax(lq.float(), dim=-1).clamp(1e-9, 1.0)
            kl = (p * (p.log() - q.log())).sum(dim=-1).mean()
            kl_vals.append(kl.item())

    return float(np.mean(kl_vals))


# ── Token accuracy ────────────────────────────────────────────────
def token_accuracy(model, enc, prompt_enc):
    """Completion token accuracy — the surface-level metric."""
    correct = total = 0
    with torch.no_grad():
        for i in range(enc['input_ids'].shape[0]):
            ids    = enc['input_ids'][i].unsqueeze(0)
            mask   = enc['attention_mask'][i].unsqueeze(0)
            plen   = int(prompt_enc['attention_mask'][i].sum().item())
            flen   = int(mask.sum().item())
            logits = model(input_ids=ids, attention_mask=mask).logits
            for t in range(plen, flen-1):
                correct += int(logits[0,t].argmax().item() == ids[0,t+1].item())
                total   += 1
    return (correct / total * 100.0) if total > 0 else 0.0


# ── Get MLP activations via hook ──────────────────────────────────
def get_mlp_activations(model, enc, layer_idx, n_prompts=100):
    """Extract MLP output activations at a specific layer."""
    acts = []
    hook = model.model.layers[layer_idx].mlp.register_forward_hook(
        lambda m, i, o: acts.append(o.detach().float().cpu())
    )
    with torch.no_grad():
        for i in range(min(n_prompts, enc['input_ids'].shape[0])):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)
            model(input_ids=ids, attention_mask=mask)
    hook.remove()
    return torch.cat(acts, dim=0)


# ── Mutual information estimator ──────────────────────────────────
def mutual_information(X, Y, n_bins=15):
    """
    Histogram-based MI: I(X;Y) = H(X) + H(Y) - H(X,Y)
    n_bins=15 for n=500 prompts (500/225 = 2.2 samples/bin)
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    X = (X - X.min()) / (X.max() - X.min() + 1e-9)
    Y = (Y - Y.min()) / (Y.max() - Y.min() + 1e-9)
    joint, _, _ = np.histogram2d(X, Y, bins=n_bins)
    joint = joint / (joint.sum() + 1e-9)
    px, py = joint.sum(axis=1), joint.sum(axis=0)
    def H(p):
        p = p[p > 1e-9]
        return float(-np.sum(p * np.log2(p)))
    return max(H(px) + H(py) - H(joint.flatten()), 0.0)


# ── Average bits ──────────────────────────────────────────────────
def avg_bits(allocation, n_layers):
    bmap = {'fp32':32,'int8':8,'int4':4}
    total = sum(bmap.get(v,32) for v in allocation.values())
    total += 32 * (n_layers - len(allocation))
    return total / n_layers


print('All utility functions defined ✓')

All utility functions defined ✓


---
## Cell 6 — EXP 1: Ground Truth KL-D Map

Quantizes each of 32 layers one at a time. Measures KL-D.
Uses in-place quantization + stored FP32 logits — no deepcopy of 8B model.

**Expected time: ~20 minutes**

In [9]:
from scipy import stats

print('=' * 55)
print('EXP 1 — Ground Truth KL-D Map')
print('=' * 55)

# ── Step 1: Store FP32 outputs once ──────────────────────────────
print('\nStep 1: Storing FP32 calibration outputs...')
cal_fp32_logits = store_fp32_outputs(model_fp, cal_enc)

# Y_task for MI estimation (used in EXP 1 Option B)
Y_task_cal = np.array([
    cal_fp32_logits[i].float().softmax(dim=-1).max(dim=-1).values.mean().item()
    for i in range(len(cal_fp32_logits))
])

# ── Step 2: Layer-wise KL-D map ──────────────────────────────────
print('\nStep 2: Layer-wise KL-D map (32 layers × 500 prompts)...')
kl_gt = []

for layer_idx in range(N_LAYERS):
    named_layers = get_layer_linears(model_fp, layer_idx)

    # Save only this layer's 7 weight matrices (~200MB per layer)
    originals = {name: layer.weight.data.clone() for name, layer in named_layers}

    # Quantize this layer to INT4
    for name, layer in named_layers:
        quantize_int4_inplace(layer)

    # Measure KL-D against stored FP32 outputs
    kl = kl_from_stored(model_fp, cal_enc, cal_fp32_logits)
    kl_gt.append(kl)

    # Restore original weights immediately
    for name, layer in named_layers:
        layer.weight.data = originals[name]

    # Free fragmented VRAM every 8 layers
    if (layer_idx + 1) % 8 == 0:
        torch.cuda.empty_cache()

    print(f'  Layer {layer_idx:2d}: KL-D = {kl:.6f}')

kl_gt = np.array(kl_gt)
print(f'\nMost degraded:  Layer {np.argmax(kl_gt)}  KL = {kl_gt.max():.5f}')
print(f'Least degraded: Layer {np.argmin(kl_gt)}  KL = {kl_gt.min():.5f}')
print(f'Ratio: {kl_gt.max()/kl_gt.min():.1f}×')

EXP 1 — Ground Truth KL-D Map

Step 1: Storing FP32 calibration outputs...
  Storing FP32 outputs (runs once, reused for all layers)...
    100/500 done
    200/500 done
    300/500 done
    400/500 done
    500/500 done
  Stored 500 logit tensors on CPU

Step 2: Layer-wise KL-D map (32 layers × 500 prompts)...
  Layer  0: KL-D = 3.546237
  Layer  1: KL-D = 1.983716
  Layer  2: KL-D = 1.303587
  Layer  3: KL-D = 1.510559
  Layer  4: KL-D = 1.295908
  Layer  5: KL-D = 1.392087
  Layer  6: KL-D = 1.237076
  Layer  7: KL-D = 1.266557
  Layer  8: KL-D = 1.129243
  Layer  9: KL-D = 1.160165
  Layer 10: KL-D = 0.902794
  Layer 11: KL-D = 0.943319
  Layer 12: KL-D = 0.916663
  Layer 13: KL-D = 0.986387
  Layer 14: KL-D = 1.139089
  Layer 15: KL-D = 1.423929
  Layer 16: KL-D = 0.970492
  Layer 17: KL-D = 0.937595
  Layer 18: KL-D = 0.905742
  Layer 19: KL-D = 0.571518
  Layer 20: KL-D = 0.891651
  Layer 21: KL-D = 0.598458
  Layer 22: KL-D = 0.491893
  Layer 23: KL-D = 0.618660
  Layer 24: KL-

---
## Cell 7 — Option A and B Scores (T-Selection)

In [10]:
print('Computing T-selection scores...')

# ── Option A: Weight Frobenius Norm ──────────────────────────────
# Theoretically invalid: weights are input-independent → I(T;X) = 0
option_a = []
for layer_idx in range(N_LAYERS):
    total_norm = sum(
        torch.norm(layer.weight.data.float(), 'fro').item()
        for _, layer in get_layer_linears(model_fp, layer_idx)
    )
    option_a.append(total_norm)
option_a = np.array(option_a)
print(f'Option A range: {option_a.min():.1f} to {option_a.max():.1f}')

# ── Option B: Activation MI I(Q(h_l); Y) ─────────────────────────
# Theoretically valid: activations are input-dependent → valid IB object
# Uses random projection to reduce 4096-dim to 64-dim before MI estimation
# Johnson-Lindenstrauss: preserves pairwise distances approximately
print('\nComputing Option B: I(Q(h_l); Y) per layer...')

option_b = []
# Fixed random projection matrix (same for all layers)
torch.manual_seed(SEED)
proj = torch.randn(HIDDEN_DIM, 64) / np.sqrt(64)

# Use first 100 prompts for activation MI (faster, enough for estimation)
Y_task_100 = Y_task_cal[:100]

for layer_idx in range(N_LAYERS):
    # Full precision activations
    acts = get_mlp_activations(model_fp, cal_enc, layer_idx, n_prompts=100)

    # Simulate INT4 quantization on activations
    scale  = acts.abs().max() / 7.0 + 1e-8
    acts_q = (acts / scale).round().clamp(-7, 7) * scale

    # Project 4096-dim → 64-dim, then L2 norm → scalar per prompt
    acts_proj = (acts_q @ proj).norm(dim=-1).mean(dim=-1).numpy()

    mi = mutual_information(acts_proj, Y_task_100)
    option_b.append(mi)

    if layer_idx % 8 == 0:
        print(f'  Layer {layer_idx:2d}: I(Q(h);Y) = {mi:.4f} bits')

option_b = np.array(option_b)

# ── Spearman Rank Correlation ─────────────────────────────────────
rho_a, p_a = stats.spearmanr(option_a, kl_gt)
rho_b, p_b = stats.spearmanr(option_b, kl_gt)

print(f'\nSpearman ρ vs Ground Truth KL-D:')
print(f'  Option A (Weight Norm):    ρ = {rho_a:.4f}  p = {p_a:.4f}  ← theoretically invalid')
print(f'  Option B (Activation MI):  ρ = {rho_b:.4f}  p = {p_b:.4f}  ← valid IB object')
print(f'\n  Winner: {"Option B ✓" if rho_b > rho_a else "Option A — investigate"}')
print(f'  On 500 prompts + 32 layers, Option B ρ expected to be positive')

Computing T-selection scores...
Option A range: 458.7 to 565.8

Computing Option B: I(Q(h_l); Y) per layer...
  Layer  0: I(Q(h);Y) = 1.2885 bits
  Layer  8: I(Q(h);Y) = 1.1094 bits
  Layer 16: I(Q(h);Y) = 1.1911 bits
  Layer 24: I(Q(h);Y) = 1.0571 bits

Spearman ρ vs Ground Truth KL-D:
  Option A (Weight Norm):    ρ = -0.6470  p = 0.0001  ← theoretically invalid
  Option B (Activation MI):  ρ = -0.0018  p = 0.9921  ← valid IB object

  Winner: Option B ✓
  On 500 prompts + 32 layers, Option B ρ expected to be positive


---
## Cell 8 — EXP 4: Layer Information Plane

In [11]:
print('EXP 4 — Layer Information Plane...')

# Input entropy per prompt (X signal)
def input_entropy(enc, n=100):
    H = []
    for i in range(min(n, enc['input_ids'].shape[0])):
        ids = enc['input_ids'][i].cpu().numpy()
        _, counts = np.unique(ids, return_counts=True)
        p = counts / counts.sum()
        H.append(float(-np.sum(p * np.log2(p + 1e-9))))
    return np.array(H)

X_input = input_entropy(cal_enc, n=100)

# Quantize entire model to INT4 for displacement measurement
print('Quantizing entire model to INT4 for EXP 4...')
all_originals = {}
for layer_idx in range(N_LAYERS):
    named = get_layer_linears(model_fp, layer_idx)
    all_originals[layer_idx] = {name: layer.weight.data.clone() for name, layer in named}
    for name, layer in named:
        quantize_int4_inplace(layer)
print('Model fully quantized to INT4')

# Measure per-layer information plane coordinates
ix_fp, iy_fp = [], []
ix_q4, iy_q4 = [], []

for layer_idx in range(N_LAYERS):
    # Temporarily restore this layer to FP to get FP32 activations
    for name, layer in get_layer_linears(model_fp, layer_idx):
        layer.weight.data = all_originals[layer_idx][name]

    af = get_mlp_activations(model_fp, cal_enc, layer_idx, n_prompts=100)
    Xf = (af @ proj).norm(dim=-1).mean(dim=-1).numpy()
    ix_fp.append(mutual_information(Xf, X_input))
    iy_fp.append(mutual_information(Xf, Y_task_cal[:100]))

    # Re-quantize back to INT4
    for name, layer in get_layer_linears(model_fp, layer_idx):
        quantize_int4_inplace(layer)

    aq = get_mlp_activations(model_fp, cal_enc, layer_idx, n_prompts=100)
    Xq = (aq @ proj).norm(dim=-1).mean(dim=-1).numpy()
    ix_q4.append(mutual_information(Xq, X_input))
    iy_q4.append(mutual_information(Xq, Y_task_cal[:100]))

    if layer_idx % 8 == 0:
        print(f'  Layer {layer_idx:2d}: FP [{ix_fp[-1]:.3f},{iy_fp[-1]:.3f}]  '
              f'INT4 [{ix_q4[-1]:.3f},{iy_q4[-1]:.3f}]')

# Restore all weights to FP32
for layer_idx in range(N_LAYERS):
    for name, layer in get_layer_linears(model_fp, layer_idx):
        layer.weight.data = all_originals[layer_idx][name]
torch.cuda.empty_cache()
print('All weights restored to FP32 ✓')

ix_fp = np.array(ix_fp);  iy_fp = np.array(iy_fp)
ix_q4 = np.array(ix_q4);  iy_q4 = np.array(iy_q4)
arrow_len = np.sqrt((ix_q4-ix_fp)**2 + (iy_q4-iy_fp)**2)

print(f'\nMost displaced:  Layer {np.argmax(arrow_len)}  '
      f'displacement = {arrow_len.max():.4f}')
print(f'Least displaced: Layer {np.argmin(arrow_len)}  '
      f'displacement = {arrow_len.min():.4f}')
print(f'Ratio: {arrow_len.max()/arrow_len.min():.1f}×')

EXP 4 — Layer Information Plane...
Quantizing entire model to INT4 for EXP 4...
Model fully quantized to INT4
  Layer  0: FP [1.378,1.115]  INT4 [1.802,1.158]
  Layer  8: FP [1.401,1.018]  INT4 [0.840,0.803]
  Layer 16: FP [1.191,1.161]  INT4 [1.258,1.175]
  Layer 24: FP [1.099,1.036]  INT4 [1.117,0.906]
All weights restored to FP32 ✓

Most displaced:  Layer 8  displacement = 0.6003
Least displaced: Layer 28  displacement = 0.0231
Ratio: 26.0×


---
## Cell 9 — Compute β and Bit Allocation

In [12]:
def compute_beta(kl_gt, arrow_len, alpha=0.5):
    """
    βₗ = α · norm(KL-Dₗ) + (1−α) · norm(displacementₗ)
    Normalized so mean(β) = 1.0
    β > 1.0 → protect (more bits)
    β < 1.0 → compress (fewer bits)
    """
    def norm(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)
    beta_raw = alpha * norm(kl_gt) + (1 - alpha) * norm(arrow_len)
    return beta_raw / (beta_raw.mean() + 1e-9)


def compute_bit_allocation(beta, target_bits, n_layers):
    """
    Sort by β. Top-k layers → INT8. Rest → INT4.
    k = n × (target - 4) / 4
    """
    sorted_layers = np.argsort(beta)[::-1]
    k = int(round(n_layers * (target_bits - 4.0) / 4.0))
    k = max(0, min(k, n_layers))
    alloc = {int(l): ('int8' if r < k else 'int4')
             for r, l in enumerate(sorted_layers)}
    actual = np.mean([8 if v=='int8' else 4 for v in alloc.values()])
    return alloc, actual


# ── α sensitivity at 6-bit ────────────────────────────────────────
print('α sensitivity analysis at 6-bit...')
print(f'{"α":>5}  INT8 layers')
print('─' * 55)

for alpha_val in [0.0, 0.2, 0.5, 0.8, 1.0]:
    b = compute_beta(kl_gt, arrow_len, alpha=alpha_val)
    alloc, _ = compute_bit_allocation(b, 6.0, N_LAYERS)
    int8 = sorted([l for l,v in alloc.items() if v=='int8'])
    marker = '← EXP 4 only' if alpha_val==0.0 else \
             '← EXP 1 only (Stage 2)' if alpha_val==1.0 else ''
    print(f'{alpha_val:>5.1f}  {int8}  {marker}')

# ── Build allocations at all bit targets ─────────────────────────
# Use α = 0.5 (update if sensitivity shows a different α is better)
alpha = 0.5
beta  = compute_beta(kl_gt, arrow_len, alpha=alpha)

print(f'\nUsing α = {alpha}')
print(f'β range: {beta.min():.3f} to {beta.max():.3f}')

all_allocations = {}
for target in [4.0, 5.0, 6.0, 7.0, 8.0]:
    alloc, actual = compute_bit_allocation(beta, target, N_LAYERS)
    all_allocations[target] = alloc
    k = sum(1 for v in alloc.values() if v=='int8')
    int8_layers = sorted([l for l,v in alloc.items() if v=='int8'])
    print(f'  {target:.0f}-bit: {k} layers INT8  actual={actual:.1f}  {int8_layers}')

α sensitivity analysis at 6-bit...
    α  INT8 layers
───────────────────────────────────────────────────────
  0.0  [0, 1, 4, 6, 7, 8, 9, 10, 12, 13, 15, 21, 23, 26, 27, 29]  ← EXP 4 only
  0.2  [0, 1, 3, 4, 6, 7, 8, 9, 10, 12, 13, 15, 21, 23, 26, 27]  
  0.5  [0, 1, 3, 4, 6, 7, 8, 9, 10, 12, 13, 14, 15, 21, 23, 26]  
  0.8  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 31]  
  1.0  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 15, 16, 30, 31]  ← EXP 1 only (Stage 2)

Using α = 0.5
β range: 0.002 to 3.283
  4-bit: 0 layers INT8  actual=4.0  []
  5-bit: 8 layers INT8  actual=5.0  [0, 1, 4, 8, 10, 12, 15, 21]
  6-bit: 16 layers INT8  actual=6.0  [0, 1, 3, 4, 6, 7, 8, 9, 10, 12, 13, 14, 15, 21, 23, 26]
  7-bit: 24 layers INT8  actual=7.0  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 20, 21, 23, 26, 27, 31]
  8-bit: 32 layers INT8  actual=8.0  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


---
## Cell 10 — Run All Experiments

In [13]:
def apply_allocation_inplace(model, allocation):
    """Apply bit allocation to model in-place. Returns saved weights."""
    saved = {}
    for layer_idx, prec in allocation.items():
        named = get_layer_linears(model, layer_idx)
        saved[layer_idx] = {name: layer.weight.data.clone() for name, layer in named}
        for name, layer in named:
            if   prec == 'int8': quantize_int8_inplace(layer)
            elif prec == 'int4': quantize_int4_inplace(layer)
    return saved

def restore_weights(model, saved):
    """Restore model weights from saved dict."""
    for layer_idx, name_weights in saved.items():
        named = dict(get_layer_linears(model, layer_idx))
        for name, weights in name_weights.items():
            named[name].weight.data = weights

# Store validation FP32 outputs
print('Storing FP32 validation outputs...')
val_fp32_logits = store_fp32_outputs(model_fp, val_enc)

# Run all configurations
print('\nRunning all experiments...')
print('=' * 60)
results = []

def evaluate(label, allocation, bits, config_type):
    print(f'\n  [{config_type}] {label}  ({bits:.1f}-bit)')
    if allocation:
        saved = apply_allocation_inplace(model_fp, allocation)
    kl_cal = kl_from_stored(model_fp, cal_enc, cal_fp32_logits)
    kl_val = kl_from_stored(model_fp, val_enc, val_fp32_logits)
    print(f'    KL(cal)={kl_cal:.5f}   KL(val)={kl_val:.5f}')
    if allocation:
        restore_weights(model_fp, saved)
        torch.cuda.empty_cache()
    results.append({'label':label,'bits':bits,'kl_cal':kl_cal,'kl_val':kl_val,'type':config_type})

# Baselines
evaluate('FP32 Baseline', {}, 32.0, 'baseline')
evaluate('Uniform INT8',  {i:'int8' for i in range(N_LAYERS)}, 8.0, 'uniform')
evaluate('Uniform INT4',  {i:'int4' for i in range(N_LAYERS)}, 4.0, 'uniform')

# IB allocations
for target, alloc in sorted(all_allocations.items()):
    actual = np.mean([8 if v=='int8' else 4 for v in alloc.values()])
    evaluate(f'IB-{target:.0f}bit (β=combined)', alloc, actual, 'ib')

print('\n' + '=' * 60)
print('All experiments complete ✓')

Storing FP32 validation outputs...
  Storing FP32 outputs (runs once, reused for all layers)...
    100/100 done
  Stored 100 logit tensors on CPU

Running all experiments...

  [baseline] FP32 Baseline  (32.0-bit)
    KL(cal)=0.00000   KL(val)=0.00000

  [uniform] Uniform INT8  (8.0-bit)
    KL(cal)=0.64758   KL(val)=0.61716

  [uniform] Uniform INT4  (4.0-bit)
    KL(cal)=10.49733   KL(val)=10.65775

  [ib] IB-4bit (β=combined)  (4.0-bit)
    KL(cal)=10.49733   KL(val)=10.65775

  [ib] IB-5bit (β=combined)  (5.0-bit)
    KL(cal)=8.71191   KL(val)=8.77184

  [ib] IB-6bit (β=combined)  (6.0-bit)
    KL(cal)=11.84526   KL(val)=12.00860

  [ib] IB-7bit (β=combined)  (7.0-bit)
    KL(cal)=2.86082   KL(val)=2.72986

  [ib] IB-8bit (β=combined)  (8.0-bit)
    KL(cal)=0.64758   KL(val)=0.61716

All experiments complete ✓


---
## Cell 11 — Final Results Summary

In [14]:
import json

print('=' * 70)
print(f'LLAMA 3.1 8B — IB QUANTIZATION RESULTS')
print(f'Model: {MODEL_NAME}')
print(f'Calibration: {N_CALIBRATION} prompts  Validation: {N_VALIDATION} prompts')
print('=' * 70)
print(f'{"Configuration":<35} {"Bits":>6} {"KL(cal)":>10} {"KL(val)":>10}')
print('─' * 65)
for r in results:
    print(f'{r["label"]:<35} {r["bits"]:>6.1f} '
          f'{r["kl_cal"]:>10.5f} {r["kl_val"]:>10.5f}')

# T-Selection
print(f'\nT-Selection:')
print(f'  Option A ρ = {rho_a:.4f}  (p={p_a:.4f})  Weight Norm — theoretically invalid')
print(f'  Option B ρ = {rho_b:.4f}  (p={p_b:.4f})  Activation MI — T = Q(h)')
winner = 'Option B ✓ — positive ρ confirms T = Q(h) at scale' if rho_b > 0 else \
         'Option B ✓ — less wrong than A' if rho_b > rho_a else 'Investigate'
print(f'  → {winner}')

# Pareto check
print(f'\nPareto check — IB vs Uniform:')
print(f'{"Config":<30} {"Bits":>5} {"KL":>10} {"Note"}')
print('─' * 65)
u8_kl = next(r['kl_cal'] for r in results if 'INT8' in r['label'])
u4_kl = next(r['kl_cal'] for r in results if 'INT4' in r['label'])
for r in results:
    if r['type'] == 'ib':
        if 4.5 <= r['bits'] <= 7.5:
            note = 'new operating point — uniform cannot do this'
        elif r['bits'] >= 7.6:
            pct = (u8_kl - r['kl_cal']) / u8_kl * 100
            note = f'{pct:+.1f}% vs uniform INT8'
        else:
            pct = (u4_kl - r['kl_cal']) / u4_kl * 100
            note = f'{pct:+.1f}% vs uniform INT4'
        print(f'{r["label"]:<30} {r["bits"]:>5.1f} {r["kl_cal"]:>10.5f}  {note}')

# Information plane summary
print(f'\nInformation Plane (EXP 4):')
print(f'  Most displaced:  Layer {np.argmax(arrow_len)}  displacement={arrow_len.max():.4f}')
print(f'  Least displaced: Layer {np.argmin(arrow_len)}  displacement={arrow_len.min():.4f}')
print(f'  Ratio: {arrow_len.max()/arrow_len.min():.1f}×  '
      f'(GPT-2 was 30.5× — larger model expected to show larger ratio)')

# Save to JSON
output = {
    'model': MODEL_NAME,
    'n_layers': N_LAYERS,
    'n_calibration': N_CALIBRATION,
    'alpha': alpha,
    'kl_gt': kl_gt.tolist(),
    'arrow_len': arrow_len.tolist(),
    'beta': beta.tolist(),
    'rho_a': float(rho_a), 'p_a': float(p_a),
    'rho_b': float(rho_b), 'p_b': float(p_b),
    'results': results,
    'most_displaced_layer': int(np.argmax(arrow_len)),
    'least_displaced_layer': int(np.argmin(arrow_len)),
    'displacement_ratio': float(arrow_len.max()/arrow_len.min()),
}
with open('llama31_8b_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nResults saved to llama31_8b_results.json ✓')
print('Send this file + the Cell 11 output as your paper results.')

LLAMA 3.1 8B — IB QUANTIZATION RESULTS
Model: meta-llama/Llama-3.1-8B-Instruct
Calibration: 500 prompts  Validation: 100 prompts
Configuration                         Bits    KL(cal)    KL(val)
─────────────────────────────────────────────────────────────────
FP32 Baseline                         32.0    0.00000    0.00000
Uniform INT8                           8.0    0.64758    0.61716
Uniform INT4                           4.0   10.49733   10.65775
IB-4bit (β=combined)                   4.0   10.49733   10.65775
IB-5bit (β=combined)                   5.0    8.71191    8.77184
IB-6bit (β=combined)                   6.0   11.84526   12.00860
IB-7bit (β=combined)                   7.0    2.86082    2.72986
IB-8bit (β=combined)                   8.0    0.64758    0.61716

T-Selection:
  Option A ρ = -0.6470  (p=0.0001)  Weight Norm — theoretically invalid
  Option B ρ = -0.0018  (p=0.9921)  Activation MI — T = Q(h)
  → Option B ✓ — less wrong than A

Pareto check — IB vs Uniform:
Config 

In [4]:
# import os, sys, subprocess

# # ── Point everything to SSD ───────────────────────────────────────
# SSD = 'E:'   # ← change to your actual drive letter
# os.environ['HF_HOME']            = f'{SSD}\\hf_cache'
# os.environ['TRANSFORMERS_CACHE'] = f'{SSD}\\hf_cache'
# os.environ['HF_DATASETS_CACHE']  = f'{SSD}\\hf_cache'
# print(f'All caches → {SSD} ✓')

# # ── Install packages ──────────────────────────────────────────────
# subprocess.run([sys.executable, '-m', 'pip', 'install',
#     'transformers', 'datasets', 'huggingface_hub',
#     'accelerate', 'scipy', 'psutil',
#     '--cache-dir', f'{SSD}\\pip_cache', '-q'
# ], check=True)
# print('Packages installed ✓')

# ── Login ─────────────────────────────────────────────────────────
from huggingface_hub import login, get_token
login(token="YOUR_HUGGINGFACE_TOKEN")
TOKEN = get_token()
print(f'Logged in ✓')

# ── Imports ───────────────────────────────────────────────────────
import torch, numpy as np, psutil, warnings
warnings.filterwarnings('ignore')

MODEL_NAME    = 'meta-llama/Llama-3.1-8B-Instruct'
DEVICE        = 'cuda'
MODEL_DTYPE   = torch.float16
N_CALIBRATION = 500
N_VALIDATION  = 100
MAX_LEN       = 128
SEED          = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# ── System check ──────────────────────────────────────────────────
print(f'\nGPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')
total = sum(torch.cuda.get_device_properties(i).total_memory
            for i in range(torch.cuda.device_count()))/1e9
print(f'Total VRAM: {total:.1f} GB')
free_ssd = psutil.disk_usage(SSD+'\\').free/1e9
print(f'SSD free:   {free_ssd:.1f} GB')
print(f'\nReady ✓')

Logged in ✓

GPU count: 0
Total VRAM: 0.0 GB


NameError: name 'SSD' is not defined

In [3]:
import sys, os

# # Point everything to SSD
# SSD = 'E:'
# os.environ['TEMP']               = f'{SSD}\\tmp'
# os.environ['TMP']                = f'{SSD}\\tmp'
# os.environ['HF_HOME']            = f'{SSD}\\hf_cache'
# os.environ['TRANSFORMERS_CACHE'] = f'{SSD}\\hf_cache'
# os.environ['HF_DATASETS_CACHE']  = f'{SSD}\\hf_cache'
# os.makedirs(f'{SSD}\\tmp', exist_ok=True)
# os.makedirs(f'{SSD}\\hf_cache', exist_ok=True)
# os.makedirs(f'{SSD}\\pip_cache', exist_ok=True)

# # Install using the exact python from this kernel
import subprocess
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'transformers', 'datasets', 'huggingface_hub',
    'accelerate', 'scipy', 'psutil',
    '--cache-dir', f'{SSD}\\pip_cache',
    '-q'
], check=True)

print(f'Python: {sys.executable}')
print('Packages installed ✓')

NameError: name 'SSD' is not defined